In [ ]:
import pandas as pd
import polars as pl
from plotnine import *

from polars import col, lit, when

import plotly.express as px

In [ ]:
def format_df_date_column(
        df: pl.DataFrame,
        date_col: str = "day",
) -> pl.DataFrame:
        
        df = (
                df.with_columns(
                        col(date_col).str.split("/").list.to_struct(fields=["Month", "Day", "Year"]).alias("unpacked")
                )
                .unnest("unpacked")
                .with_columns(
                        (lit("20") + col("Year")).alias("Year"),
                )
                .with_columns(
                        col("Month", "Day", "Year").cast(pl.Int64),
                        col(date_col).str.to_date("%m/%d/%y").alias("Date")
                )
                .with_columns(
                        pl.col("Date").dt.weekday().alias("DayOfWeek"),
                        pl.col("Date").dt.strftime("%A").alias("DayOfWeekStr"),
                )
                .with_columns(
                        when(col("DayOfWeek").is_in([6, 7]))
                        .then(lit("Weekend"))
                        .otherwise(lit("Weekday"))
                        .alias("WeekdayOrWeekend")
                )
                .rename({"day": "DayStr",})
        )
        return df


In [ ]:
df_mu = pl.read_csv("Data/Monthly_Listeners.txt", separator="\t")
df_du = pl.read_csv("Data/Daily_Listeners.txt", separator="\t")
df_ds = pl.read_csv("Data/Daily_Seconds_Listened.txt", separator="\t")

df_mu = (
    df_mu.with_columns(
        (
            pl.col("Year").cast(pl.Utf8) 
            + "-" 
            + pl.col("Month").cast(pl.Utf8).str.zfill(2)
        )
        .alias("YearMonth")
    )
    .rename({"Listeners": "NumListeners"})
)

df_du = (
    format_df_date_column(df_du)
    .rename({
        "platform": "Platform",
        "listeners": "NumListeners", 
    })
)

df_ds = (
    format_df_date_column(df_ds)
    .rename({
        "platform": "Platform",
        "seconds": "NumSecondsListened", 
    })
)


display(df_du, df_mu, df_ds)


In [ ]:
df_du_monthly_sum = (
    df_du.group_by("Platform", "Month", "Year")
    .agg(col("NumListeners").sum().alias("TotalDailyUniques"))
)

df_du_monthly_sum


df_mu_du = (
    df_mu.join(
        df_du_monthly_sum,
        on=["Platform", "Month", "Year"],
        how="left"
    )
    .rename({"NumListeners": "MonthlyUniques"})
    .with_columns(
        (col("TotalDailyUniques")/col("MonthlyUniques")).alias("AvgDaysListened")
    )
)

df_mu_du.filter(
    (col("Month") == 12)
    & (col("Year") == 2019)
).select(
    col("MonthlyUniques").sum(),
    col("TotalDailyUniques").sum(),
)

# df_du

# df_du.describe()

# df_mu_du

In [ ]:
df_ds
df_du_ds = (
    df_du.join(
        df_ds, 
        on=["DayStr", "Platform", "Month", "Day", "Year", "Date", "DayOfWeek", "DayOfWeekStr", "WeekdayOrWeekend"], 
        how="left"
    )
    .with_columns((col("NumSecondsListened") / col("NumListeners")).alias("SecondsPerListener"))
)

df_du_ds


In [ ]:
(
    ggplot(data=df_ds, mapping=aes(x="Date", y="NumSecondsListened", fill="Platform", color="Platform"))
    + geom_line()
)

(
    ggplot(data=df_du_ds, mapping=aes(x="Date", y="SecondsPerListener/60", fill="Platform", color="Platform"))
    + geom_line()
    # + geom_smooth(method="loess")
    # + scale_x_date(date_breaks="1 day", date_labels="%a")
    # + coord_cartesian(xlim=[pd.to_datetime("2018-01-01"), pd.to_datetime("2018-01-30")])
    + theme_bw()
    + theme(
       figure_size=[5, 4] 
    )
)

In [ ]:
(
    ggplot(data=df_du, mapping=aes(x="Date", y="NumListeners", fill="Platform", color="Platform"))
    + geom_line()
    + scale_x_date(date_breaks="1 day", date_labels="%a")
    + coord_cartesian(xlim=[pd.to_datetime("2018-01-01"), pd.to_datetime("2018-01-30")])
    + theme_bw()
    + theme(
       figure_size=[14, 6] 
    )
)

In [ ]:
(
    df_du.group_by("DayOfWeekStr")
    .agg(col("NumListeners").mean())
)

df_du_by_day_of_week = (
    df_du.group_by("Platform", "DayOfWeek", "DayOfWeekStr")
    .agg(
        col("NumListeners").mean().alias("NumListenersMean"),
        col("NumListeners").std().alias("NumListenersStd"),
    )
)

df_du_by_weekend = (
    df_du.group_by("Platform", "WeekdayOrWeekend")
    .agg(
        col("NumListeners").mean().alias("NumListenersMean"),
        col("NumListeners").std().alias("NumListenersStd"),
    )
)

display(df_du_by_day_of_week, df_du_by_weekend)
(
    ggplot(data=df_du_by_day_of_week, mapping=aes(x="DayOfWeek", y="NumListenersMean"))
    + facet_wrap("Platform")
    + geom_col()
    + geom_errorbar(mapping=aes(ymin="NumListenersMean-NumListenersStd", ymax="NumListenersMean+NumListenersStd"))
)

(
    ggplot(data=df_du_by_weekend, mapping=aes(x="WeekdayOrWeekend", y="NumListenersMean"))
    + facet_wrap("Platform", scales="free_y")
    + geom_col()
    + geom_errorbar(mapping=aes(ymin="NumListenersMean-NumListenersStd", ymax="NumListenersMean+NumListenersStd"))
)

In [ ]:
df_du_by_weekend

In [ ]:
df_du_by_weekend_pct_change = (
    df_du_by_weekend.pivot(
        on="WeekdayOrWeekend",\
        index="Platform",
        values="NumListenersMean"
    )
    .select("Platform", "Weekday", "Weekend")
    .with_columns(
        (
            (col("Weekend") - col("Weekday")) 
            / col("Weekday")
        ).alias("PctChangeWeekendVsWeekday")
    )
    .with_columns(
        (col("PctChangeWeekendVsWeekday") > 0).alias("IsGainOnWeekend"),
        (
            (col("PctChangeWeekendVsWeekday") * 100)
                .round(1)
                .cast(pl.String) 
                + lit("%")
        ).alias("PctChangeLabel")
    )
)

order_platform_by_pct_change = (
    df_du_by_weekend_pct_change.select(["Platform", "PctChangeWeekendVsWeekday"])
    # .unique(subset=["Platform"])
    .sort("PctChangeWeekendVsWeekday", descending=True)
    .get_column("Platform")
    .to_list()
)


df_du_by_weekend_pct_change = df_du_by_weekend_pct_change.with_columns(
    col("Platform").cast(pl.Enum(categories=order_platform_by_pct_change))
)

df_du_by_weekend_pct_change

(
    ggplot(
        data=df_du_by_weekend_pct_change, 
        mapping=aes(x="Platform", y="PctChangeWeekendVsWeekday", fill="IsGainOnWeekend")
    )
    # + facet_wrap("Platform", scales="free_y")
    + geom_col()
    + geom_text(mapping=aes(label="PctChangeLabel"), nudge_y=.025, size=10)
    + scale_fill_manual({True: "green", False: "red"})
)

# platform_order_by_pct_change
# df_du_by_weekend_pct_change


In [ ]:
fig = px.bar(
    df_du_by_weekend_pct_change,
    x="Platform",
    y="PctChangeWeekendVsWeekday",
    color="IsGainOnWeekend",
    text="PctChangeLabel",
    color_discrete_map={True: "green", False: "red"},
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:

# Sort so bars stack/order correctly along the x-axis
df_mu = df_mu.sort(["Year", "Month"])
display(df_mu)
# Create stacked bar chart
fig = px.bar(
    df_mu,
    x="YearMonth",
    y="NumListeners",
    color="Platform",
    category_orders={"Platform": ["iOS", "Android",  "Web", "All_Other", "Roku", "Amazon",]}
)

fig.update_layout(
    barmode="stack", 
    bargap=0,
    xaxis_title="Year-Month", 
    yaxis_title="NumListeners",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray",
    gridwidth=1,
    griddash="dot",
    layer="above traces",
)

fig.show()

In [ ]:
fig.update_yaxes(
    range=[55_000_000, 70_000_000]
)

fig.show()

In [ ]:
fig.write_html(
    "test_plot.html",
    include_plotlyjs="cdn",
    full_html=True
)